# 1. Add Similarity to Pretrain Dataset

## Goal
- Add **cosine similarity** based indices and distances to **pretrain dataset parquet files**


## Features
1. **Efficiency**: Load data once, compute similarity
2. **Top-K**: Extract top 20 indices and distances for indexing


## Configuration and Paths
- `datasets/pretrain/pretrain_pairs_ctx512`
    - https://huggingface.co/datasets/nkh/TS-RAG-Data/tree/main/pretrain_pairs_ctx512
- `retrieval_database/retrieval_database_512.parquet`
    - https://huggingface.co/datasets/nkh/TS-RAG-Data/blob/main/retrieval_database_512.parquet

In [ ]:
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import glob
import os
import gc
from sim_utils import *

# GPU setup
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.device_count() >= 2:
    print(f"Available GPUs: {torch.cuda.device_count()}")
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")
    if torch.cuda.device_count() > 1:
        print(f"GPU 1: {torch.cuda.get_device_name(1)}")

In [ ]:
PRETRAIN_DATA_DIR = "datasets/pretrain/pretrain_pairs_ctx512"
RETRIEVAL_DATABASE_PATH = "retrieval_database/retrieval_database_512.parquet"

parquet_files = sorted(glob.glob(os.path.join(PRETRAIN_DATA_DIR, "*.parquet")))
print(f"Found {len(parquet_files)} parquet files")

In [ ]:
# Settings
K = 20  # Top-K number
INPUT_LEN = 512  # Input TS length (use first 512 points for similarity calculation)
BATCH_SIZE = 50000  # GPU batch size (adjust based on GPU memory)
USE_GPU = True  # Use GPU

In [ ]:
database_df = pd.read_parquet(RETRIEVAL_DATABASE_PATH, engine='pyarrow')
database_x = database_df['x'].values
database_array = np.array([np.array(x[:INPUT_LEN], dtype=np.float32) for x in database_x], dtype=np.float32)
print(f"Database shape: {database_array.shape}")
print(f"Database memory usage: {database_array.nbytes / 1024**3:.2f} GB")

In [ ]:
def extract_input_ts_from_file(file_path):
    """Extract input TS from file as numpy array"""
    df = pd.read_parquet(file_path, columns=['target'], engine='pyarrow')
    targets = df['target'].values
    input_ts = np.array([np.array(ts[:INPUT_LEN], dtype=np.float32) for ts in targets], dtype=np.float32)
    return input_ts, len(df)

In [ ]:
# Extract input TS from all files
query_ts_list = []
file_sizes = []  

for i, file in enumerate(tqdm(parquet_files, desc="Extracting Query TS")):
    input_ts, n_rows = extract_input_ts_from_file(file)
    query_ts_list.append(input_ts)
    file_sizes.append(n_rows)

query_array = np.vstack(query_ts_list)

del query_ts_list
gc.collect()

print(f"\nQuery shape: {query_array.shape}")
print(f"Memory usage: {query_array.nbytes / 1024**3:.2f} GB")
print(f"Total Query rows: {sum(file_sizes):,}")


In [ ]:
# Compute top-K indices and distances
indices_list, distances_list = compute_topk_indices_distances_cosine(
    query_array,  # Query: extracted from pretrain parquet files
    database_array,  # Database: 'x' column from retrieval_database
    k=K,
    batch_size=BATCH_SIZE,
    gpu_id=0,
    device=device,
    USE_GPU=USE_GPU
)

# Convert to numpy arrays for saving
indices_array = np.array([np.array(idx) for idx in indices_list])
distances_array = np.array([np.array(dist) for dist in distances_list])

In [ ]:
results_file = os.path.join('datasets/pretrain', f"similarity_T{INPUT_LEN}.npz")

np.savez_compressed(
    results_file,
    indices=indices_array,
    distances=distances_array,
    config_name='pretrain_similarity',
    k=K,
    n_samples=len(indices_list)
)

del indices_array, distances_array

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
results = np.load(results_file)

indices_array = results['indices']
distances_array = results['distances']

indices_list = [indices_array[i] for i in range(len(indices_array))]
distances_list = [distances_array[i] for i in range(len(distances_array))]

In [ ]:
new_dir = f"{PRETRAIN_DATA_DIR}_X_space"
os.makedirs(new_dir, exist_ok=True)

current_idx = 0
for file_path in tqdm(parquet_files, desc=f"Saving files"):
    # Read original file
    df = pd.read_parquet(file_path, engine='pyarrow')
    n_rows = len(df)
    
    # Extract indices and distances for current file
    file_indices = indices_list[current_idx:current_idx + n_rows]
    file_distances = distances_list[current_idx:current_idx + n_rows]
    
    # Add indices and distances columns (keep other columns unchanged)
    df["indices"] = file_indices
    df["distances"] = file_distances
    
    # Create new file path
    original_filename = os.path.basename(file_path)
    new_file_path = os.path.join(new_dir, original_filename)
    
    # Save to new directory
    df.to_parquet(new_file_path, engine='pyarrow', index=False)
    
    current_idx += n_rows
    
    # Clear memory
    del df
    gc.collect()

print(f" Complete! Saved {len(parquet_files)} files")